# Data retrieval from MUL site for Mech Assist application
This code retrieves the whole list of units from http://masterunitlist.info/ and their Alpha Strike parameters.

In [ ]:
ПОИСК = "Uller (Kit Fox)"
ШАССИ = "Uller (Kit Fox)"

BattleMech = True;
Combat_Vehicle = False;
Aerospace = False;
Infantry = False;
IndustrialMech = False;
Protomech = False;
Support_Vehicle = False;
Advanced_Aerospace = False;
Advanced_Support = False;
Building = False;
Unknown = False;

types = [t for t, enabled in [
    ("&Types=18", BattleMech),
    ("&Types=19", Combat_Vehicle),
    ("&Types=17", Aerospace),
    ("&Types=21", Infantry),
    ("&Types=20", IndustrialMech),
    ("&Types=23", Protomech),
    ("&Types=24", Support_Vehicle),
    ("&Types=81", Advanced_Aerospace),
    ("&Types=79", Advanced_Support),
    ("&Types=97", Building),
    ("&Types=76", Unknown)
] if enabled]

search_url = "http://www.masterunitlist.info/Unit/Filter?Name=" + ПОИСК.replace(" ", "+") + "&MinPV=1&MaxPV=100"
search_url += search_url.join(types)
print(search_url)

In [ ]:
!python -m pip install tqdm
!python -m pip install pandas
!python -m pip install openpyxl
import requests
from bs4 import BeautifulSoup
import csv
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm
import openpyxl
from openpyxl.utils import get_column_letter

Connect and get confirmation of connection

In [ ]:
# Send an HTTP GET request to the URL
response = requests.get(search_url)
if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    print("Access granted!")
else:
    print("Failed to retrieve the webpage.")

Make a list of all mechs

In [ ]:
unit_names = [
    a.text for a in soup.find_all('a', href=True)
    if a['href'].startswith("/Unit/Details/")
]

unit_urls = [
    "https://masterunitlist.info" + a['href']
    for a in soup.find_all('a', href=True)
    if a['href'].startswith("/Unit/Details/")
]

paired = list(zip(unit_names, unit_urls))

paired.sort(key=lambda x: int(''.join(filter(str.isdigit, x[0]))) if ''.join(filter(str.isdigit, x[0])) else 0)

unit_names = [name for name, url in paired]
unit_urls  = [url  for name, url in paired]

print(unit_names)

Make a dataframe to collect units parameters

In [ ]:
dataset = pd.DataFrame(columns = ["Name", "Model", "Type", "Role", "Era", "Tech", "Specials", "PV", "Size", "TMM", "Move", "Jump", "Short", "Medium", "Long", "Overheat", "Armor", "Structure", "Tonnage"])

Fill `dataset` with units' information:

In [ ]:
Jump = 0
TMM = 0
MoveI = 0
EraI = 0
Era = "STAR LEAGUE"

for i, unit_name in enumerate(unit_names):
    if (i%50 == 0):
        percent_complete = (i + 1) / len(unit_names) * 100

    # Define the URL with the query parameters
    url = f"https://masterunitlist.azurewebsites.net/Unit/QuickList?Name={unit_name}"

    # Send an HTTP GET request with the parameters
    response = requests.get(url, stream=True)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:

        # Parse the JSON response
        data = response.json()

        # Identify which unit should be parsed as one request may return several units. For example:
        # https://masterunitlist.azurewebsites.net/Unit/QuickList?Name=Phoenix%20Hawk%20PXH-1k

        for item in data.get("Units"):
            if (item.get("Name")==unit_name):
                index = data.get("Units").index(item)

                Move = data.get("Units")[index].get("BFMove", "")
                Move = Move.replace('"',"")
                if "j" in Move:
                    Move = Move.replace('j',"")
                    if "/" in Move:
                        MoveS = Move.split("/")
                        Jump = MoveS[1]
                        Move = MoveS[0]
                    else:
                        Jump = Move
                else:
                    MoveI = Move

        Move = numbers = "".join([char for char in Move if char.isdigit()])
        MoveI = int(Move)
        match MoveI:
            case n if 4 <= n <= 8:
                TMM = 1
            case n if 9 <= n <= 12:
                TMM = 2
            case n if 13 <= n <= 18:
                TMM = 3
            case n if 19 <= n <= 34:
                TMM = 4
            case n if 35 <= n <= 1000:
                TMM = 5
            case _:
                TMM = "?"

        EraI = int(data.get("Units")[index].get("EraId"))
        match EraI:
            case n if n == 9 or n == 10:
                Era = "STAR LEAGUE"
            case n if n == 11 or n == 255 or n == 256:
                Era = "SUCCESSION WARS"
            case n if n == 13:
                Era = "CLAN INVASION"
            case n if n == 247:
                Era = "CIVIL WAR"
            case n if n == 14:
                Era = "JIHAD"
            case n if n == 15 or n == 254 or n == 16:
                Era = "DARK AGE"
            case n if n == 257:
                Era = "ILCLAN"
            case _:
                Era = data.get("Units")[index].get("EraId")

        Tech = data.get("Units")[index].get("Technology").get("Name")
        match Tech:
          case n if n == "Clan":
              Tech = "CLAN"
          case n if n == "Inner Sphere":
              Tech = "IS"
          case n if n == "Primitive":
              Tech = "PRIMITIVE"
          case _:
              Tech = "OTHER"

        Size = data.get("Units")[index].get("BFSize", 0)
        match Size:
          case n if n == 1:
              Size = "1L"
          case n if n == 2:
              Size = "2M"
          case n if n == 3:
              Size = "3H"
          case n if n == 4:
              Size = "4A"
          case _:
              Size = "OTHER"

        # Extract the desired information
        parsed_unit = {
            "Name": ШАССИ,
            "Model": data.get("Units")[index].get("Name"),
            "Type": data.get("Units")[index].get("Type").get("Name").upper(),
            "Role": data.get("Units")[index].get("Role").get("Name").upper(),
            "Era": Era,
            "Tech": Tech,
            "Specials": data.get("Units")[index].get("BFAbilities", ""),
            "PV": data.get("Units")[index].get("BFPointValue", 0),
            "Size": Size,
            "TMM": TMM,
            "Move": Move,
            "Jump": Jump,
            "Short": data.get("Units")[index].get("BFDamageShort", 0),
            "Medium": data.get("Units")[index].get("BFDamageMedium", 0),
            "Long": data.get("Units")[index].get("BFDamageLong", 0),
            "Overheat": data.get("Units")[index].get("BFOverheat", 0),
            "Armor": data.get("Units")[index].get("BFArmor", 0),
            "Structure": data.get("Units")[index].get("BFStructure", 0),
            "Tonnage": data.get("Units")[index].get("FormatedTonnage")
        }

        print(parsed_unit)

        # Add unit parameters to the dataframe
        dataset = pd.concat([dataset, pd.DataFrame([parsed_unit])], ignore_index=True)

print(f"Data has been saved to 'datase' DataFrame")

Clear the dataset and make a new one (to have an access to initial data).
All the units with 0 PV can be dropped as they shouldn't be used during a game

In [ ]:
unitlist = dataset[dataset['PV'] != 0].reset_index(drop=True)
#unitlist.info()

Check all NA values and replace them if applicable

In [ ]:
unitlist[unitlist["Model"].isna()].head()

Specials can be None

In [ ]:
unitlist[unitlist["Specials"].isna()].head()

In [ ]:
path = f"unit_list_{ШАССИ}.xlsx"

unitlist.to_excel(
    path,
    index=False,
    engine='openpyxl'
)

wb = openpyxl.load_workbook(path)
ws = wb.active

from openpyxl.styles import Font, Alignment

data_font = Font(name="Roboto", size=10, color="434343")
hyperlink_font = Font(name="Roboto", size=10, color="1155cc", underline="single")
data_alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)

for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
    for cell in row:
        cell.font = data_font
        cell.alignment = data_alignment

model_col = None
for col in range(1, ws.max_column + 1):
    if ws.cell(row=1, column=col).value == "Model":
        model_col = col
        break

if model_col is not None:
    for row_idx in range(2, ws.max_row + 1):
        cell = ws.cell(row=row_idx, column=model_col)

        idx = row_idx - 2
        if idx < len(unit_urls):
            full_url = unit_urls[idx]
        else:
            full_url = None

        if full_url:
            cell.hyperlink = full_url
            cell.font = hyperlink_font

wb.save(path)